# Simulation Experiment Demo Code

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io
from scipy.optimize import fsolve, linear_sum_assignment
from scipy.special import expit
from scipy.stats import t
from sklearn.metrics import roc_auc_score, mean_squared_error, adjusted_rand_score
import gc
from ALMA import AltMin
from balm import run_balm_hurdle_nuts_from_ZY, BALMHyperParams

# Coupling Experiments

In [ ]:
def generate_hurdle_network(rng_key, L, n, M, K, a0_star, a1_star, sigma_star=1.0, alpha=1.0):
    key_u, key_gamma, key_w, key_z, key_y = random.split(rng_key, 5)
    
    # Generate orthonormal bases and score matrices
    U = random.normal(key_u, (M, n, K))
    U = jnp.linalg.qr(U)[0] 
    gamma = random.normal(key_gamma, (M, K))
    
    Q = jnp.zeros((M, n, n))
    for m in range(M):
        S_m = U[m] @ jnp.diag(gamma[m]) @ U[m].T
        Q = Q.at[m].set(S_m - jnp.diag(jnp.diag(S_m)))
        
    # Generate mixing weights
    dirichlet_alpha = jnp.full((M,), alpha / M)
    W = random.dirichlet(key_w, dirichlet_alpha, (L,))
    
    # Compute latent means
    mu = jnp.einsum('lm,mij->lij', W, Q)
    
    # Hurdle observation model
    logits = a0_star + a1_star * mu
    prob = jax.nn.sigmoid(logits)
    
    Z = random.bernoulli(key_z, prob)
    Y_noise = random.normal(key_y, (L, n, n)) * sigma_star
    Y = mu + Y_noise
    
    # Mask diagonal and lower triangle to prevent leakage
    mask = jnp.triu(jnp.ones((n, n)), k=1)
    Z = Z * mask
    Y = Y * mask
    
    return Z, Y, Q, W

def objective_a0(a0, a1_star, target_density, rng_key, L, n, M, K):
    _, _, Q, W = generate_hurdle_network(rng_key, L, n, M, K, a0_star=0.0, a1_star=a1_star)
    mu = jnp.einsum('lm,mij->lij', W, Q)
    logits = a0 + a1_star * mu
    prob = jax.nn.sigmoid(logits)
    mask = jnp.triu(jnp.ones((n, n)), k=1)
    expected_density = jnp.sum(prob * mask) / (L * n * (n - 1) / 2)
    return float(expected_density - target_density)

def find_a0_for_density(target_density, a1_star, rng_key, L, n, M, K):
    initial_guess = -2.0
    a0_opt, = fsolve(objective_a0, initial_guess, args=(a1_star, target_density, rng_key, L, n, M, K))
    return a0_opt

def align_and_permute(Q_est, Q_true, W_est=None):
    M = Q_true.shape[0]
    cost_matrix = np.zeros((M, M))
    for i in range(M):
        for j in range(M):
            corr = np.corrcoef(Q_est[i].flatten(), Q_true[j].flatten())[0, 1]
            cost_matrix[i, j] = -corr

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    avg_corr = float(-cost_matrix[row_ind, col_ind].mean())

    Q_aligned = np.zeros_like(Q_est)
    for est_idx, true_idx in zip(row_ind, col_ind):
        Q_aligned[true_idx] = Q_est[est_idx]

    if W_est is not None:
        W_aligned = np.zeros_like(W_est)
        for est_idx, true_idx in zip(row_ind, col_ind):
            W_aligned[:, true_idx] = W_est[:, est_idx]
        return Q_aligned, W_aligned, avg_corr

    return Q_aligned, avg_corr

In [ ]:
L_val, n_val, M_val, K_val = 100, 68, 3, 4
num_replications = 15
a1_stars = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
densities = [0.15, 0.30, 0.50]

results = []
master_key = random.PRNGKey(42)

for density in densities:
    for a1_true in a1_stars:
        
        corr_coupled_list = []
        corr_decoupled_list = []
        a1_est_list = []
        
        for rep in range(num_replications):
            master_key, key_data, key_fit1, key_fit2 = random.split(master_key, 4)
            
            # Calibrate baseline intercept
            a0_calibrated = find_a0_for_density(density, a1_true, key_data, L_val, n_val, M_val, K_val)
            
            Z_data, Y_data, Q_true, _ = generate_hurdle_network(
                key_data, L_val, n_val, M_val, K_val, a0_star=a0_calibrated, a1_star=a1_true
            )
            
            # Fit Coupled BALM (fully specified)
            hyper_coupled = BALMHyperParams(a1_fixed=None)
            mcmc_coupled = run_balm_hurdle_nuts_from_ZY(
                rng_key=key_fit1, Z=Z_data, Y=Y_data, n=n_val, M=M_val, K=K_val,
                hyper=hyper_coupled, num_warmup=1000, num_samples=1500, progress_bar=True
            )
            samples_coupled = mcmc_coupled.get_samples()
            Q_est_coupled = jnp.mean(samples_coupled['Q'], axis=0)
            a1_est = jnp.mean(samples_coupled['a1'])
            
            # Fit Decoupled BALM (restricted, a1=0)
            hyper_decoupled = BALMHyperParams(a1_fixed=0.0)
            mcmc_decoupled = run_balm_hurdle_nuts_from_ZY(
                rng_key=key_fit2, Z=Z_data, Y=Y_data, n=n_val, M=M_val, K=K_val,
                hyper=hyper_decoupled, num_warmup=1000, num_samples=1500, progress_bar=True
            )
            samples_decoupled = mcmc_decoupled.get_samples()
            Q_est_decoupled = jnp.mean(samples_decoupled['Q'], axis=0)
            
            corr_coupled = align_and_permute(np.array(Q_est_coupled), np.array(Q_true))
            corr_decoupled = align_and_permute(np.array(Q_est_decoupled), np.array(Q_true))
            
            corr_coupled_list.append(corr_coupled)
            corr_decoupled_list.append(corr_decoupled)
            a1_est_list.append(float(a1_est))
   
        results.append({
            'Target_Density': density,
            'True_a1': a1_true,
            'Coupled_Mean': np.mean(corr_coupled_list),
            'Coupled_Std': np.std(corr_coupled_list),
            'Decoupled_Mean': np.mean(corr_decoupled_list),
            'Decoupled_Std': np.std(corr_decoupled_list),
            'Estimated_a1_Mean': np.mean(a1_est_list)
        })

df_results = pd.DataFrame(results)
df_results.to_csv('sparsity_coupling_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
offset = 0.05 

for i, density in enumerate(densities):
    df_subset = df_results[df_results['Target_Density'] == density]
    
    axes[i].errorbar(df_subset['True_a1'], df_subset['Coupled_Mean'], yerr=df_subset['Coupled_Std'], fmt='-o', label='Coupled BALM', capsize=5, elinewidth=2, color='blue', linewidth=2)
    axes[i].errorbar(df_subset['True_a1'] + offset, df_subset['Decoupled_Mean'], yerr=df_subset['Decoupled_Std'], fmt='--s', label='Decoupled (a1=0)', capsize=5, elinewidth=2, color='red', linewidth=2)
    axes[i].set_title(f'Template Recovery (Density = {int(density*100)}%)', fontsize=14)
    axes[i].set_xlabel('True Coupling Strength (a1)', fontsize=12)
    if i == 0:
        axes[i].set_ylabel('Template Correlation', fontsize=12)
    
    axes[i].set_ylim(0, 1.05)
    axes[i].legend(loc='upper left')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ALMA VS BALM

In [ ]:
def apply_edge_masking(A_true, missing_rate=0.15, seed=123):
    """
    Randomly masks a percentage of upper-triangular edges for out-of-sample prediction.
    """
    rng = np.random.default_rng(seed)
    n_subjects, n_nodes, _ = A_true.shape
    A_train = A_true.copy()
    test_mask = np.zeros((n_subjects, n_nodes, n_nodes), dtype=bool)

    triu_idx = np.triu_indices(n_nodes, k=1)
    n_edges_per_layer = len(triu_idx[0])
    n_mask_per_layer = int(n_edges_per_layer * missing_rate)

    for ell in range(n_subjects):
        permuted_indices = rng.permutation(n_edges_per_layer)
        mask_idx = permuted_indices[:n_mask_per_layer]
        row_idx = triu_idx[0][mask_idx]
        col_idx = triu_idx[1][mask_idx]

        test_mask[ell, row_idx, col_idx] = True
        test_mask[ell, col_idx, row_idx] = True

        # Mask continuous and topological information
        A_train[ell, row_idx, col_idx] = np.nan
        A_train[ell, col_idx, row_idx] = np.nan

    return A_train, test_mask

def calculate_balm_posterior_mean_Q(samples, M, N):
    """
    Extracts the posterior mean of the off-diagonal templates Q_m from BALM samples.
    """
    Q_mean = np.zeros((M, N, N))
    U_samp = np.asarray(samples['U'])
    gamma_samp = np.asarray(samples['gamma'])
    tau_samp = np.asarray(samples['tau'])
    n_draws = U_samp.shape[0]

    for m in range(M):
        Q_m_sum = np.zeros((N, N))
        for i in range(n_draws):
            U_im = U_samp[i, m]
            gamma_im = gamma_samp[i, m]
            tau_i = tau_samp[i]
            Q_im = tau_i * (U_im @ np.diag(gamma_im) @ U_im.T)
            np.fill_diagonal(Q_im, 0.0)
            Q_m_sum += Q_im
        Q_mean[m] = Q_m_sum / n_draws
    return Q_mean


def rel_frobenius_error(true_mat, pred_mat, mask):
    return float(np.linalg.norm((true_mat - pred_mat)[mask]) / (np.linalg.norm(true_mat[mask]) + 1e-9))

In [ ]:
def run_mixed_membership_replicate(alpha_val, rep, L_subj=500, N_nodes=68, M_true=3, K_true=3):
    print(f"\n--- Running: Alpha = {alpha_val} | Replication = {rep+1} ---")
    current_seed = 42 + int(alpha_val * 100) + rep
    rng_key_gen = random.PRNGKey(current_seed)

    # Generate unified Data via the Coupling generator
    Z_full_jax, Y_full_jax, Q_true_jax, W_true_jax = generate_hurdle_network(
        rng_key=rng_key_gen, L=L_subj, n=N_nodes, M=M_true, K=K_true, 
        a0_star=-2.5, a1_star=5.0, sigma_star=1.0, alpha=alpha_val
    )
    
    Z_full = np.array(Z_full_jax)
    A_full = np.array(Y_full_jax) * Z_full  # Combined topology and weights
    Q_true = np.array(Q_true_jax)
    W_true = np.array(W_true_jax)

    A_train, test_mask = apply_edge_masking(A_full, missing_rate=0.15, seed=current_seed)
    A_train_filled = np.nan_to_num(A_train, nan=0.0)

    # Export dataset on the first replication
    if rep == 0:
        export_filename = f"sim_data_alpha_{alpha_val}_rep_{rep}.mat"
        scipy.io.savemat(export_filename, {
            "A_full": A_full,
            "Z_true": Z_full,
            "test_mask": test_mask,
            "Q_true": Q_true,
            "W_true": W_true
        })

    # ALMA Model (Topology-Only Baseline)
    Z_train_alma = np.where(A_train_filled > 0, 1.0, 0.0)
    alma_params = {'K': np.full(M_true, K_true), 'L': L_subj, 'M': M_true, 'n': N_nodes, 'max_iter': 5000, 'verbose': False}
    Q_alma, W_alma = AltMin(Z_train_alma, alma_params)

    # BALM Model (Fully Generative)
    Z_fit = jnp.array(np.where(A_train_filled > 0, 1, 0))
    Y_fit = jnp.array(A_train_filled)

    hyper = BALMHyperParams(use_student_t=False, alpha=alpha_val)
    key_fit = random.PRNGKey(current_seed + 1)

    samples = run_balm_hurdle_nuts_from_ZY(
        rng_key=key_fit, Z=Z_fit, Y=Y_fit, n=N_nodes, M=M_true, K=K_true,
        hyper=hyper, num_warmup=500, num_samples=500, num_chains=1, progress_bar=False
    )

    # Process BALM
    Q_balm_raw = calculate_balm_posterior_mean_Q(samples, M_true, N_nodes)
    W_balm_raw = np.asarray(samples['W']).mean(axis=0)

    # Align templates via Hungarian algorithm
    Q_alma_aligned, W_alma_aligned, alma_corr = align_and_permute(Q_alma, Q_true, W_est=W_alma)
    Q_balm_aligned, W_balm_aligned, balm_corr = align_and_permute(Q_balm_raw, Q_true, W_est=W_balm_raw)

    Z_test_true = Z_full[test_mask]
    A_test_true = A_full[test_mask]
    nonzero_test_mask = (Z_test_true == 1)

    # ALMA predictions (Topology reconstruction)
    Z_alma_recon = np.einsum('lm, mjk -> ljk', W_alma_aligned, Q_alma_aligned)
    alma_preds = Z_alma_recon[test_mask]
    auc_alma = roc_auc_score(Z_test_true, alma_preds)
    
    mse_alma = mean_squared_error(A_test_true[nonzero_test_mask], alma_preds[nonzero_test_mask])
    frob_alma = rel_frobenius_error(A_full, Z_alma_recon, test_mask)

    # BALM predictions
    a0_mean = float(np.asarray(samples['a0']).mean())
    a1_mean = float(np.asarray(samples['a1']).mean())
    mu_balm = np.einsum('lm, mjk -> ljk', W_balm_aligned, Q_balm_aligned)
    balm_pi = expit(a0_mean + a1_mean * mu_balm)
    
    # Expected continuous values conditionally on edge presence
    balm_A_recon = mu_balm 
    balm_expected_A = balm_pi * balm_A_recon

    auc_balm = roc_auc_score(Z_test_true, balm_pi[test_mask])
    mse_balm = mean_squared_error(A_test_true[nonzero_test_mask], balm_A_recon[test_mask][nonzero_test_mask])
    frob_balm = rel_frobenius_error(A_full, balm_expected_A, test_mask)

    # Fair ARI Calculation (Uniform Argmax for both)
    y_true = np.argmax(W_true, axis=1)
    y_balm = np.argmax(W_balm_raw, axis=1)
    y_alma = np.argmax(W_alma, axis=1)

    ari_alma = adjusted_rand_score(y_true, y_alma)
    ari_balm = adjusted_rand_score(y_true, y_balm)

    del samples, Q_balm_raw, W_balm_raw, Z_fit, Y_fit, A_train_filled
    jax.clear_caches()
    gc.collect()

    return {
        "Alpha": alpha_val, "Rep": rep,
        "ALMA_AUC": auc_alma, "BALM_AUC": auc_balm,
        "ALMA_MSE": mse_alma, "BALM_MSE": mse_balm,
        "ALMA_Frob": frob_alma, "BALM_Frob": frob_balm,
        "ALMA_ARI": ari_alma, "BALM_ARI": ari_balm,
        "ALMA_Corr": alma_corr, "BALM_Corr": balm_corr
    }

In [ ]:
alphas_to_test = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 3.0]

n_reps = 50 
all_mm_results = []

for alpha in alphas_to_test:
    for rep in range(n_reps):
        res = run_mixed_membership_replicate(alpha, rep=rep, L_subj=500)
        all_mm_results.append(res)

df_mm_raw = pd.DataFrame(all_mm_results)

df_mm_agg = df_mm_raw.groupby("Alpha").agg(['mean', 'std']).reset_index()
df_mm_agg.columns = ['_'.join(col).strip('_') for col in df_mm_agg.columns.values]

print("\n=== FINAL MIXED-MEMBERSHIP EXPERIMENT RESULTS ===")
print(df_mm_agg.to_string(index=False))
df_mm_agg.to_csv("balm_vs_alma_results_final.csv", index=False)

# Appendix A.2: Heavy-Tailed Robustness

In [ ]:
def generate_heavy_tailed_network(rng_key, L, n, M, K, a0_star, a1_star, nu, sigma_star=1.0, alpha=1.0):
    key_u, key_gamma, key_w, key_z = random.split(rng_key, 4)
    
    U = random.normal(key_u, (M, n, K))
    U = jnp.linalg.qr(U)[0] 
    gamma = random.normal(key_gamma, (M, K))
    
    Q = jnp.zeros((M, n, n))
    for m in range(M):
        S_m = U[m] @ jnp.diag(gamma[m]) @ U[m].T
        Q = Q.at[m].set(S_m - jnp.diag(jnp.diag(S_m)))
        
    dirichlet_alpha = jnp.full((M,), alpha / M)
    W = random.dirichlet(key_w, dirichlet_alpha, (L,))
    mu = jnp.einsum('lm,mij->lij', W, Q)
    
    logits = a0_star + a1_star * mu
    prob = jax.nn.sigmoid(logits)
    Z = random.bernoulli(key_z, prob)
    
    Y_noise = t.rvs(df=nu, scale=sigma_star, size=(L, n, n), random_state=np.random.default_rng(42))
    Y = mu + Y_noise
    
    mask = jnp.triu(jnp.ones((n, n)), k=1)
    Z = Z * mask
    Y = Y * mask
    
    return np.array(Z), np.array(Y), np.array(Q), np.array(W)

def run_heavy_tail_robustness_experiment():
    L_val, n_val, M_val, K_val = 100, 68, 3, 4
    nu_values = [3, 5, 10]
    
    results = []
    master_key = random.PRNGKey(202)
    
    for nu in nu_values:
        key_data, key_fit_gauss, key_fit_t, master_key = random.split(master_key, 4)
        
        Z_data, Y_data, Q_true, W_true = generate_heavy_tailed_network(
            key_data, L=L_val, n=n_val, M=M_val, K=K_val, 
            a0_star=-2.5, a1_star=5.0, nu=nu
        )
        
        hyper_gauss = BALMHyperParams(use_student_t=False)
        mcmc_gauss = run_balm_hurdle_nuts_from_ZY(
            rng_key=key_fit_gauss, Z=jnp.array(Z_data), Y=jnp.array(Y_data), 
            n=n_val, M=M_val, K=K_val, hyper=hyper_gauss, 
            num_warmup=500, num_samples=500, progress_bar=False
        )
        samples_gauss = mcmc_gauss.get_samples()
        Q_gauss_raw = calculate_balm_posterior_mean_Q(samples_gauss, M_val, n_val)
        W_gauss_raw = np.asarray(samples_gauss['W']).mean(axis=0)
        
        hyper_t = BALMHyperParams(use_student_t=True, t_df=nu)
        mcmc_t = run_balm_hurdle_nuts_from_ZY(
            rng_key=key_fit_t, Z=jnp.array(Z_data), Y=jnp.array(Y_data), 
            n=n_val, M=M_val, K=K_val, hyper=hyper_t, 
            num_warmup=500, num_samples=500, progress_bar=False
        )
        samples_t = mcmc_t.get_samples()
        Q_t_raw = calculate_balm_posterior_mean_Q(samples_t, M_val, n_val)
        W_t_raw = np.asarray(samples_t['W']).mean(axis=0)
        
        _, _, corr_gauss = align_and_permute(Q_gauss_raw, W_gauss_raw, Q_true)
        _, _, corr_t = align_and_permute(Q_t_raw, W_t_raw, Q_true)
        
        results.append({'True_Nu': nu, 'Model': 'Gaussian', 'Template_Corr': corr_gauss})
        results.append({'True_Nu': nu, 'Model': 'Student-t', 'Template_Corr': corr_t})

    df_results = pd.DataFrame(results)
    df_results.to_csv("appendix_heavy_tail_robustness.csv", index=False)
    return df_results

In [ ]:
df_heavy_tail = run_heavy_tail_robustness_experiment()

df_ht_agg = df_heavy_tail.groupby(['True_Nu', 'Model']).agg(['mean', 'std']).reset_index()
df_ht_agg.columns = ['_'.join(col).strip('_') for col in df_ht_agg.columns.values]
df_ht_agg = df_ht_agg.rename(columns={'True_Nu_': 'True_Nu', 'Model_': 'Model'})
print(df_ht_agg.to_string(index=False))

# Appendix A.3: Template Misspecification

In [ ]:
def calculate_waic(Z, Y, a0_mean, a1_mean, mu_samples, sigma_mean):
    S = mu_samples.shape[0]
    log_lik_samples = np.zeros((S, Z.size))
    
    Z_flat = Z.flatten()
    Y_flat = Y.flatten()
    
    for s in range(S):
        mu_flat = mu_samples[s].flatten()
        logits_s = a0_mean + a1_mean * mu_flat
        prob_s = jax.nn.sigmoid(logits_s)
        
        ll_Z = Z_flat * np.log(prob_s + 1e-9) + (1 - Z_flat) * np.log(1 - prob_s + 1e-9)
        
        ll_Y = np.zeros_like(Z_flat)
        mask = Z_flat == 1
        ll_Y[mask] = -0.5 * np.log(2 * np.pi * sigma_mean**2) - 0.5 * ((Y_flat[mask] - mu_flat[mask])**2) / sigma_mean**2
        
        log_lik_samples[s] = ll_Z + ll_Y
        
    lppd = np.sum(np.log(np.mean(np.exp(log_lik_samples), axis=0)))
    p_waic = np.sum(np.var(log_lik_samples, axis=0))
    waic = -2 * (lppd - p_waic)
    
    return waic

def run_m_misspecification_experiment():
    L_val, n_val, M_true, K_val = 100, 68, 5, 4
    M_fits = [2, 4, 5, 6, 8, 10]
    
    master_key = random.PRNGKey(101)
    key_data, master_key = random.split(master_key)
    
    Z_data, Y_data, _, _ = generate_hurdle_network(
        key_data, L=L_val, n=n_val, M=M_true, K=K_val, 
        a0_star=-2.5, a1_star=5.0, sigma_star=1.0, alpha=1.0
    )
    
    Z_train, test_mask = apply_edge_masking(np.array(Z_data), missing_rate=0.15, seed=101)
    Y_train = np.array(Y_data).copy()
    Y_train[test_mask] = 0.0
    
    Z_test_true = np.array(Z_data)[test_mask]
    results = []
    
    for M_fit in M_fits:
        key_fit, master_key = random.split(master_key)
        hyper = BALMHyperParams(use_student_t=False)
        
        mcmc = run_balm_hurdle_nuts_from_ZY(
            rng_key=key_fit, Z=jnp.array(Z_train), Y=jnp.array(Y_train), 
            n=n_val, M=M_fit, K=K_val, hyper=hyper, 
            num_warmup=500, num_samples=500, progress_bar=False
        )
        
        samples = mcmc.get_samples()
        a0_mean = np.mean(samples['a0'])
        a1_mean = np.mean(samples['a1'])
        sigma_mean = np.mean(samples['sigma'])
        
        W_samp = np.asarray(samples['W'])
        U_samp = np.asarray(samples['U'])
        gamma_samp = np.asarray(samples['gamma'])
        tau_samp = np.asarray(samples['tau'])
        
        S_draws = W_samp.shape[0]
        mu_samples = np.zeros((S_draws, L_val, n_val, n_val))
        
        for s in range(S_draws):
            for m in range(M_fit):
                Q_m = tau_samp[s] * (U_samp[s, m] @ np.diag(gamma_samp[s, m]) @ U_samp[s, m].T)
                np.fill_diagonal(Q_m, 0.0)
                mu_samples[s] += np.einsum('l,ij->lij', W_samp[s, :, m], Q_m)
                
        mu_mean = np.mean(mu_samples, axis=0)
        waic_val = calculate_waic(Z_train, Y_train, a0_mean, a1_mean, mu_samples, sigma_mean)
        
        logits_pred = a0_mean + a1_mean * mu_mean
        pi_pred = jax.nn.sigmoid(logits_pred)
        auc_val = roc_auc_score(Z_test_true, pi_pred[test_mask])
        
        results.append({
            'Fitted_M': M_fit,
            'WAIC': waic_val,
            'AUC': auc_val
        })
        
    df_results = pd.DataFrame(results)
    df_results.to_csv("appendix_m_misspecification_results.csv", index=False)
    return df_results

In [ ]:
df_m_misspec = run_m_misspecification_experiment()
print(df_m_misspec.to_string(index=False))